# 4. Comparación de Desempeño: HOG vs SIFT vs FREAK
### Evaluación Comparativa en la Tarea de Image Detection / Classification

En este cuaderno se comparan frente a frente los tres métodos en la tarea de **Image Detection (Clasificación de Imágenes: dada una imagen, decir a qué clase pertenece)** sobre el mismo conjunto de prueba de 140 imágenes:
1. **HOG + SVM**
2. **SIFT + BoVW + SVM**
3. **FREAK + BoVW + SVM**



In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Cargar resumen numérico del benchmark
with open("benchmark_summary.json", "r") as f:
    results = json.load(f)

methods = list(results.keys())
accuracies = [results[m]["Accuracy (%)"] for m in methods]
latencies = [results[m]["Latency (ms/img)"] for m in methods]
fps_vals = [results[m]["Throughput (FPS)"] for m in methods]
sizes = [results[m]["Model Size (MB)"] for m in methods]

print("=" * 80)
print(f"{'Método':<8} | {'Exactitud':<12} | {'Latencia (ms)':<14} | {'FPS':<10} | {'Tamaño Modelo':<12}")
print("-" * 80)
for m in methods:
    print(f"{m:<8} | {results[m]['Accuracy (%)']:>10.2f}% | {results[m]['Latency (ms/img)']:>12.2f} ms | {results[m]['Throughput (FPS)']:>10.1f} | {results[m]['Model Size (MB)']:>10.2f} MB")
print("=" * 80)



## Gráficas Comparativas de Rendimiento


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
colors = ['#1f77b4', '#9467bd', '#ff7f0e']

# 1. Exactitud
axes[0].bar(methods, accuracies, color=colors, width=0.55)
axes[0].set_title("Exactitud en Prueba (%)", fontsize=12, fontweight='bold')
axes[0].set_ylim(0, 100)
axes[0].grid(axis='y', linestyle='--', alpha=0.5)
for i, v in enumerate(accuracies):
    axes[0].text(i, v + 2, f"{v:.2f}%", ha='center', fontweight='bold')

# 2. Latencia
axes[1].bar(methods, latencies, color=colors, width=0.55)
axes[1].set_title("Latencia de Inferencia (ms/imagen)", fontsize=12, fontweight='bold')
axes[1].grid(axis='y', linestyle='--', alpha=0.5)
for i, v in enumerate(latencies):
    axes[1].text(i, v + 0.8, f"{v:.2f} ms", ha='center', fontweight='bold')

# 3. Tamaño del Modelo
axes[2].bar(methods, sizes, color=colors, width=0.55)
axes[2].set_title("Tamaño del Modelo (MB)", fontsize=12, fontweight='bold')
axes[2].grid(axis='y', linestyle='--', alpha=0.5)
for i, v in enumerate(sizes):
    axes[2].text(i, v + 0.9, f"{v:.2f} MB", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()



## Visualización de Muestras Clasificadas por los Tres Descriptores


In [ ]:
import cv2

pred_demo_path = Path("visualizations/classification_predictions_demo.png")
if pred_demo_path.exists():
    img_demo = cv2.imread(str(pred_demo_path))
    plt.figure(figsize=(15, 10))
    plt.imshow(cv2.cvtColor(img_demo, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("Ejemplos de Clasificación de Imágenes (Dada una imagen -> Clase predicha)", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()



## Resumen Comparativo de Arquitectura para Clasificación de Imágenes

| Dimensión | HOG + Linear SVM | SIFT + BoVW + SVM | FREAK + BoVW + SVM |
| :--- | :--- | :--- | :--- |
| **Representación de la Imagen** | Vector global denso de gradientes ($128 \times 128$) | Histograma BoVW ($150$ palabras visuales) | Histograma BoVW ($150$ palabras visuales) |
| **Dimensión por Imagen** | 8,100 valores flotantes | 150 valores flotantes | 150 valores flotantes |
| **Exactitud en Prueba** | **84.29%** | 70.00% | 71.43% |
| **Velocidad de Inferencia** | **0.73 ms** (1,377 FPS) | 35.34 ms (28 FPS) | **2.93 ms** (341 FPS, **12x más rápido que SIFT**) |
| **Tamaño del Modelo** | 39.38 MB | 0.21 MB | **0.17 MB** |

